In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
footprint_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/csv/footprint/archr_footprint.csv'
footprint = pd.read_csv(footprint_path, sep='\t', index_col=0)
footprint.shape

In [ ]:
footprint.head()

In [ ]:
tfs = footprint['TF'].unique()

plot_list = []

for tf in tfs:
    tmp_footprint = footprint[footprint['TF'] == tf].copy().drop(columns=['TF'])
    tmp_footprint['cluster'] = [x.split('.')[0] for x in tmp_footprint.index]
    mean_footprint = tmp_footprint.groupby('cluster').mean().astype(int)
    mean_footprint['cluster'] = mean_footprint.index
    to_plot = mean_footprint.melt(id_vars='cluster')
    fig, ax = plt.subplots(1, figsize=(20, 8))
    ax = sns.lineplot(data=to_plot, x='variable', y='value', hue='cluster')
    ax.set_xticks([]);
    ax.set_xticklabels([]);
    ax.set_title(tf, fontsize=30)
    plot_list.append((fig, ax))
    plt.close(fig)

In [ ]:
for i,tmp in enumerate(plot_list):
    fig = tmp[0]
    ax = tmp[1]
    display(fig)

In [ ]:
tf = 'KLF4'

tmp_footprint = footprint[footprint['TF'] == tf].copy().drop(columns=['TF'])
tmp_footprint['cluster'] = [x.split('.')[0] for x in tmp_footprint.index]
mean_footprint = tmp_footprint.groupby('cluster').mean().astype(int)
mean_footprint['cluster'] = mean_footprint.index
to_plot = mean_footprint.melt(id_vars='cluster')
to_plot['value_smooth'] = (
    to_plot.groupby('cluster')['value']
    .transform(lambda x: x.rolling(7, center=True, min_periods=1).mean())
)
to_plot.head()

In [ ]:
fig, ax = plt.subplots(1, figsize=(20, 8))
ax = sns.lineplot(data=to_plot, x='variable', y='value', hue='cluster')
ax.set_xticks([]);
ax.set_xticklabels([]);
ax.set_title(tf, fontsize=30)

In [ ]:
fig, ax = plt.subplots(1, figsize=(20, 8))
ax = sns.lineplot(data=to_plot, x='variable', y='value_smooth', hue='cluster')
ax.set_xticks([]);
ax.set_xticklabels([]);
ax.set_title(tf, fontsize=30)

### just a few clusters

In [ ]:
plot_list = []

for tf in tfs:
    tmp_footprint = footprint[footprint['TF'] == tf].copy().drop(columns=['TF'])
    tmp_footprint['cluster'] = [x.split('.')[0] for x in tmp_footprint.index]
    mean_footprint = tmp_footprint.groupby('cluster').mean().astype(int)
    mean_footprint['cluster'] = mean_footprint.index
    ###
    whitelist = ['Start', 'Delayed', 'Fail1', 'Sendai', 'Inter1']
    mask = [x in whitelist for x in mean_footprint.index]
    mean_footprint = mean_footprint[mask]
    ###
    to_plot = mean_footprint.melt(id_vars='cluster')
    fig, ax = plt.subplots(1, figsize=(20, 8))
    ax = sns.lineplot(data=to_plot, x='variable', y='value', hue='cluster')
    ax.set_xticks([]);
    ax.set_xticklabels([]);
    ax.set_title(tf, fontsize=30)
    plot_list.append((fig, ax))
    plt.close(fig)

In [ ]:
for i,tmp in enumerate(plot_list):
    fig = tmp[0]
    ax = tmp[1]
    display(fig)

In [ ]:
tfs = ['POU5F1', 'SOX2', 'KLF4', 'MYC']
df_list = []

for tf in tfs:
    tmp_footprint = footprint[footprint['TF'] == tf].copy().drop(columns=['TF'])
    tmp_footprint['cluster'] = [x.split('.')[0] for x in tmp_footprint.index]
    mean_footprint = tmp_footprint.groupby('cluster').mean().astype(int)
    mean_footprint['cluster'] = mean_footprint.index
    ###
    whitelist = ['Start', 'Delayed', 'Fail1', 'Sendai', 'Inter1']
    mask = [x in whitelist for x in mean_footprint.index]
    mean_footprint = mean_footprint[mask]
    ###
    to_plot = mean_footprint.melt(id_vars='cluster')
    to_plot['value_smooth'] = (
    to_plot.groupby('cluster')['value']
    .transform(lambda x: x.rolling(7, center=True, min_periods=1).mean())
    )
    
    fig, ax = plt.subplots(1, figsize=(20, 8))
    ax = sns.lineplot(data=to_plot, x='variable', y='value_smooth', hue='cluster')
    ax.set_xticks([]);
    ax.set_xticklabels([]);
    ax.set_title(tf, fontsize=30)
    
    to_plot['TF'] = tf
    df_list.append(to_plot)

In [ ]:
merged = pd.concat(df_list)
merged.shape

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
merged.to_csv(f'{outdir}/atac_oskm_footprint.csv', sep='\t')

In [ ]:
!date